## 代码库和函数导入

In [ ]:
# 代码库倒入
import pandas as pd
import os
import joblib
import shap
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer, mean_absolute_percentage_error
import warnings
plt.rcdefaults()
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12 # 可选：设置默认字体大小
plt.rcParams['mathtext.fontset'] = 'stix'  # 数学公式字体
plt.rcParams['figure.dpi'] = 300

## 数据质量控制

In [25]:
# 合并两个时期的文件
file_202312_202402 = '/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202312-202402/DataSet.xlsx'
file_202404_202405 = '/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/DataSet.xlsx'

data_202312_202402 = pd.read_excel(file_202312_202402)
data_202404_202405 = pd.read_excel(file_202404_202405)

# 纵向合并（按行合并）
data_combined = pd.concat([data_202312_202402, data_202404_202405], axis=0, ignore_index=True)

# 按 time 列排序
data_combined = data_combined.sort_values(by='Time').reset_index(drop=True)

# 保存到新文件
data_combined.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/Dataset.xlsx', index=False)


In [ ]:
# 质量控制
print(f'剔除前：{len(data_combined)}')

# 质量控制:去尖峰、剔除仪器阴影区数据、限制稳定度参数|z/L|<2、剔除低风速(<1m/s)数据
# mask1 = (data_combined['u10'] >= 1) & (data_combined['u10'] <= 50) & (data_combined['z/l'] >= -2) & (data_combined['z/l'] <= 2) & (data_combined['charnock'] > 0) & (data_combined['charnock'] < 10)
mask1 = (data_combined['u10'] >= 0) & (data_combined['u10'] <= 50) & (data_combined['z/l'] >= -2) & (data_combined['z/l'] <=2) & (data_combined['charnock'] < 10)

data = data_combined[mask1]

print(f'剔除后：{len(data)},剔除 {len(data_combined)-len(data)}')
data.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx',index=False)

In [ ]:
#统计
file =  "/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx"
features = ['u_star_down','u10','Hs', 'direction','wave_steepnessdou', 'B*','z/l','cp','kp','fp_comp']

# 读取数据
df = pd.read_excel(file)

# 统计每列的平均值，标准差，最小值，最大值
stat_df = df[features].agg(['mean', 'std', 'min', 'max']).T
stat_df = stat_df.rename(columns={'mean': '均值', 'std': '标准差', 'min': '最小值', 'max': '最大值'})

print("各特征统计信息：")
print(stat_df)

## 绘制质量指控后，描述图

In [ ]:
# 绘制质量控制描述图
mask2= (data['u10_upper'] >= 1) & (data['u10_upper'] <= 50)  & (data['charnock'] >=0)
r2 = r2_score(data[mask2]['u10_upper'], data[mask2]['u10'])
mae = mean_absolute_error(data[mask2]['u10_upper'], data[mask2]['u10'])
print(f'r2={r2},mae={mae}')

plt.figure(figsize=(10, 5))
# 绘制原始散点图
plt.scatter(data['u10'], data['u_star_down'], marker='.', s=2, alpha=0.7, color='black', label='Quality-controlled Data')
plt.scatter(data_combined[~mask1]['u10'], data_combined[~mask1]['u_star_down'], marker='.', s=2, alpha=1, color='red', label='Removed data')

# 设置坐标轴
plt.ylim(0, 0.5)
plt.yticks(np.arange(0,0.55,0.05))
plt.xlim(0, 20)
plt.xticks(range(0,22,2))
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.xlabel('$U_{{10N}}$(m/s)',fontsize=12)
plt.ylabel(fr'$U_{{*}}$',fontsize=12)  
plt.legend(framealpha=False)  
plt.show()
plt.close()

plt.figure(figsize=(5, 5))
# 绘制原始散点图
plt.scatter(data[mask2]['u10'], data[mask2]['u10_upper'], marker='.', s=2, alpha=0.7, color='black',label='Observations')
plt.plot([0, 20], [0, 20], 'r--', linewidth=1, alpha=0.5, label='1:1 line')
# 设置坐标轴
plt.ylim(0, 20)
plt.yticks(range(0,22,2))
plt.xlim(0, 20)
plt.xticks(range(0,22,2))
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.xlabel('$U_{{10N:DOWN}}$(m/s)',fontsize=12)
plt.ylabel('$U_{{10N:UPPER}}$(m/s)',fontsize=12)

text_str = f'$R^2 = {r2:.2f}$\n$MAE = {mae:.2f}$ m/s'
plt.text(0.05, 0.95, text_str, transform=plt.gca().transAxes, verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.3))

plt.legend(framealpha=False)
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 设置分箱间隔
bin_width = 1.0
bins = np.arange(1, data['u10'].max() + bin_width, bin_width)

# 创建分箱标签
data['u10_bin'] = pd.cut(data['u10'], bins, right=False)

# 计算每个分箱的统计量
bin_stats = data.groupby('u10_bin').agg(
    u10_mean=('u10', 'mean'),
    Cdn10_mean=('Cdn10', 'mean'),
    Cdn10_std=('Cdn10', 'std'),
    count=('Cdn10', 'count')
).reset_index()

# 删除 NaN 分箱（没有数据的分箱）
bin_stats = bin_stats.dropna()

# 将 Cdn10 转换为千分之一（乘以 1000）用于绘图
Cdn10_mean_1000 = bin_stats['Cdn10_mean']
Cdn10_std_1000 = bin_stats['Cdn10_std']

# 绘图
plt.figure(figsize=(10, 5))

# 绘制原始散点图
plt.scatter(data['u10'], data['Cdn10'], marker='.', s=2, alpha=0.7, color='black')

# 绘制分箱平均值（带误差棒）
plt.errorbar(bin_stats['u10_mean'], Cdn10_mean_1000, 
             yerr=Cdn10_std_1000, 
             fmt='o', color='#A81807', markersize=4, 
             capsize=3, capthick=1, elinewidth=1,
             label=f'1m/s bins')

# 设置坐标轴
plt.yscale('log')
plt.ylim(0.0001, 0.01)
plt.xlim(0, 16)
plt.xticks(range(0,18,2))
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.xlabel(f'$U_{{10N}}$',fontsize=12)
plt.ylabel(fr'$C_{{DN}}$',fontsize=12)  
plt.legend(framealpha=False)  
plt.show()

# 打印分箱统计结果
print("分箱统计结果:")
print(bin_stats[['u10_bin', 'Cdn10_mean', 'Cdn10_std', 'count']].to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 设置分箱间隔
bin_width = 1.0
bins = np.arange(0, data['u10'].max() + bin_width, bin_width)

# 创建分箱标签
data['u10_bin'] = pd.cut(data['u10'], bins, right=False)

# 计算每个分箱的统计量 
bin_stats = data.groupby('u10_bin').agg(
    u10_mean=('u10', 'mean'),
    Cdn10_mean=('u_star_down', 'mean'),
    Cdn10_std=('u_star_down', 'std'),
    count=('u_star_down', 'count')
).reset_index()

# 删除 NaN 分箱（没有数据的分箱）
bin_stats = bin_stats.dropna()

# 将 Cdn10 转换为千分之一（乘以 1000）用于绘图
Cdn10_mean_1000 = bin_stats['Cdn10_mean']
Cdn10_std_1000 = bin_stats['Cdn10_std']

# 绘图
plt.figure(figsize=(8, 4))

# 绘制原始散点图
plt.scatter(data['u10'], data['u_star_down'], marker='.', s=5, alpha=0.7, color='#4D4D4D')

# 绘制分箱平均值（带误差棒）
plt.errorbar(np.arange(0,12,1)+0.5, Cdn10_mean_1000, 
             yerr=Cdn10_std_1000, 
             fmt='o', color='#C00000', markersize=5, 
             capsize=3, capthick=1, elinewidth=1.5,
             label=f'1m/s bins')
plt.scatter(data_combined[~mask1]['u10'], data_combined[~mask1]['u_star_down'], marker='.', s=5, alpha=1, color='#8B0000')

# 写U_star样本数据
for item in range(0,12,1):
    number = bin_stats['count'].to_numpy()[item]
    plt.text(item+0.5,0.465,f'{number}',ha='center')

# 设置坐标轴
plt.ylim(0, 0.5)
plt.yticks(np.arange(0,0.55,0.05))
plt.xlim(0, 16)
plt.xticks(range(0,17,1))
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.xlabel(f'$U_{{10N}}(m/s)$',fontsize=12)
plt.ylabel(fr'$U_{{*}}(m/s)$',fontsize=12) 
plt.legend(framealpha=False) 
plt.tight_layout()
plt.savefig("/share/home/jinhm/PhD_Project/B7_WAVE_U/image/U_star_S.tiff",dpi=300)
plt.show()

# 打印分箱统计结果
print("分箱统计结果:")
print(bin_stats[['u10_bin', 'Cdn10_mean', 'Cdn10_std', 'count']].to_string())

## 基础模型R2评估散点图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import joblib
from sklearn.metrics import r2_score, mean_squared_error


files = '/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx'
data =pd.read_excel(files)

# feature_cols = ['u10','Hs', 'direction','wave_steepnessdou', 'B*','z/l']
feature_cols = ['u10','Hs', 'direction','wave_steepnessdou', 'B*', 'cp', 'kp', 'fp_comp']
target_col = 'u_star_down'

X = data[feature_cols]
y = data[target_col]

# 划分训练集和测试集
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
models_name = [
    'xgboost',
    'lightgbm',
    'random_forest',
    'adaboost',
    'catboost',
    'svm',
    'gbdt',
    'cart',
    'plsr',
    'mlp'
]

models_name_dict = {
    'xgboost':'XGBoost',
    'lightgbm':'LightGBM',
    'random_forest':'Random Forest',
    'adaboost':'AdaBoost',
    'catboost':'CatBoost',
    'svm':'SVR',
    'gbdt':'GBDT',
    'cart':'CART',
    'plsr':'PLSR',
    'mlp':'MLP'
}

for i, name in enumerate(models_name):

    # ===== 1. 加载模型 =====
    model_path = f'/share/home/jinhm/PhD_Project/B7_WAVE_U/result/{name}/best_model.pkl'
    if not os.path.exists(model_path):
        continue
    model = joblib.load(model_path)

    # ===== 2. 预测 =====
    pre_test = model.predict(X_test)

    # ===== 3. 计算指标 =====
    r2_test = r2_score(y_test, pre_test)
    rmse = np.sqrt(mean_squared_error(y_test, pre_test))
    mape = mean_absolute_percentage_error(y_test, pre_test) * 100
    print(f'{name}--{r2_test:.3f}--{rmse:.2f}--{mape:.2f}')

    ax = axes[i]

    # ===== 4. 画散点 =====
    ax.scatter(y_test, pre_test, marker='o', s=18, alpha=0.5, facecolors='black',edgecolors='white',label='Test Set')
    # 拟合 pre_test = a * y_test  + b
    a, b = np.polyfit(y_test, pre_test, 1)
    ax.plot([0, 0.5], [a*0+b, a*0.5+b], 'r--', linewidth=1, alpha=0.5, label='Fitted line')
    # 1:1线
    ax.plot([0, 0.5], [0, 0.5], '--', color='black', linewidth=1, alpha=1, label='1:1 line')

    # 坐标范围
    ax.set_xlim(0, 0.5)
    ax.set_ylim(0, 0.5)
    ax.set_yticks(np.arange(0,0.6,0.1))
    ax.set_xticks(np.arange(0,0.6,0.1))  
    # 网格
    ax.grid(True, which='both', linestyle='--', alpha=0.3)
    #调整刻度
    ax.xaxis.set_ticks_position('both')  # x轴刻度在底部和顶部
    ax.yaxis.set_ticks_position('both')  # y轴刻度在左侧和右侧
    ax.tick_params(direction='in')
    # ===== 5. 标题 =====
    ax.set_title(f'({chr(i+97)}) {models_name_dict[name]}', fontsize=12,fontweight='bold')

    # ===== 6. 指标文本 =====
    text_str = (
                    f'y = {a:.3f}x + {b:.3f}\n'
                    f'RMSE = {rmse:.2f} m/s\n'
                    f'MAPE = {mape:.2f}%\n'
                    f'$R^2$ = {r2_test:.3f}\n'
                    f'N = {len(y_test)}'
                )
    ax.text(0.05, 0.95, text_str,
            transform=ax.transAxes,
            verticalalignment='top',
            fontsize=10,
            linespacing=1.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5,linewidth=0))

    # ===== 7. 只保留边缘坐标轴标签（更论文风格）=====
    if i // 5 == 1:  # 最下面一行
        ax.set_xlabel(r'Observed $u_*$ (m s$^{-1}$)', fontsize=12)
    else:
        ax.set_xticklabels([])

    if i % 5 == 0:  # 最左一列
        ax.set_ylabel(r'Predicted $u_*$ (m s$^{-1}$)', fontsize=12)
    else:
        ax.set_yticklabels([])
    if i == 0:
        ax.legend(framealpha=False,fontsize=10)
# ===== 8. 调整布局 =====
plt.tight_layout()
plt.subplots_adjust(wspace=0.08, hspace=0.15)
# plt.savefig("/share/home/jinhm/PhD_Project/B7_WAVE_U/image/BaseModels_Scatter.tiff",dpi=300)
plt.show()

In [2]:
# 评估已保存Meta模型(HuberRegressor)的精度，输入数据为10个一级模型的预测结果

models_name_dict = {
    'xgboost':'XGBoost',
    'lightgbm':'LightGBM',
    'random_forest':'Random Forest',
    'adaboost':'AdaBoost',
    'catboost':'CatBoost',
    'svm':'SVM',
    'gbdt':'GBDT',
    'cart':'CART',
    'plsr':'PLSR',
    'mlp':'MLP'
}

# 读取数据
data = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx')
# X = data[['u10','Hs', 'direction','wave_steepnessdou', 'B*', 'cp', 'kp', 'fp_comp']]
X = data[['u10','Hs', 'direction','wave_steepnessdou', 'B*','z/l']]
y = data['u_star_down']

# 切分测试集
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# 生成10个一级模型在测试集上的预测输出，作为Meta模型输入
base_model_preds = []
for name in models_name_dict.keys():
    model_path = f'/share/home/jinhm/PhD_Project/B7_WAVE_U/result/{name}/best_model.pkl'
    base_model = joblib.load(model_path)
    base_model_preds.append(base_model.predict(X_test))
base_model_preds = np.column_stack(base_model_preds)

# 加载已训练好的集合(HuberRegressor)模型
meta_model = joblib.load('/share/home/jinhm/PhD_Project/B7_WAVE_U/result/HuberRegressor/best_model.pkl')
meta_pred = meta_model.predict(base_model_preds)

# 计算并打印评价指标
meta_r2 = r2_score(y_test, meta_pred)
meta_rmse = np.sqrt(mean_squared_error(y_test, meta_pred))
meta_mape = mean_absolute_percentage_error(y_test, meta_pred) * 100

print(f'HuberRegressor(Stacking) results: R2={meta_r2:.3f}, RMSE={meta_rmse:.4f}, MAPE={meta_mape:.2f}%')
# HuberRegressor(Stacking) results: R2=0.801, RMSE=0.0339, MAPE=14.90%

HuberRegressor(Stacking) results: R2=0.903, RMSE=0.0234, MAPE=10.36%


In [ ]:
# 读取数据
from COARE35.coare35vn import coare35vn
# 设置全局字体和样式
plt.rcParams.update({
    'axes.labelsize': 14,  # x轴和y轴标签字体大小
    'xtick.labelsize': 14,  # x轴刻度字体大小
    'ytick.labelsize': 14,  # y轴刻度字体大小
})

models_name_dict = {
    'xgboost':'XGBoost',
    'lightgbm':'LightGBM',
    'random_forest':'Random Forest',
    'adaboost':'AdaBoost',
    'catboost':'CatBoost',
    'svm':'SVM',
    'gbdt':'GBDT',
    'cart':'CART',
    'plsr':'PLSR',
    'mlp':'MLP'
}


data = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx')
X = data[['u10','Hs', 'direction','wave_steepnessdou', 'B*', 'cp', 'kp', 'fp_comp']]
y = data['u_star_down']

# 切分测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 生成10个一级模型在测试集上的预测输出，作为Meta模型输入
base_model_preds = []
for name in models_name_dict.keys():
    model_path = f'/share/home/jinhm/PhD_Project/B7_WAVE_U/result/{name}/best_model.pkl'
    base_model = joblib.load(model_path)
    base_model_preds.append(base_model.predict(X_test))
base_model_preds = np.column_stack(base_model_preds)

# 加载已训练好的集合(HuberRegressor)模型
meta_model = joblib.load('/share/home/jinhm/PhD_Project/B7_WAVE_U/result/HuberRegressor/best_model.pkl')
meta_pred = meta_model.predict(base_model_preds)

ustar = coare35vn(
    X_test['u10'].values,
    np.full(X_test['u10'].shape, 10),
    np.full(X_test['u10'].shape, 75),
    np.full(X_test['u10'].shape, 10)
)

U_star_coare35 = ustar[:,0]

from numpy.polynomial import Polynomial
# 构建三阶多项式拟合 u10 与 u_star_down 的关系，输出系数
u10_train = X_train['u10'].values
y_star_train = y_train.values

# 拟合三阶多项式
coeffs = np.polyfit(u10_train, y_star_train, deg=3)
# 使用三阶多项式系数对 X_test['u10'] 进行预测

vickers = coeffs[0]*X_test['u10']**3 + coeffs[1]*X_test['u10']**2 + coeffs[2]*X_test['u10'] + coeffs[3]
# 三阶多项式表达式格式：coeffs[0]*x^3 + coeffs[1]*x^2 + coeffs[2]*x + coeffs[3]
# vickers = 0.17-0.019*X_test['u10']+0.0042*X_test['u10']**2-8.4e-5*X_test['u10']**3

# 为保证代码可复现，假定相关变量已经定义（见上述上下文）
# meta_pred: 集成模型预测值
# U_star_coare35: COARE35计算的u*
# vickers: Vickers公式的u*
# y_test: 观测u*（标签）

# 以u10风速分bin统计RMSE
wind_col = 'u10'
bin_width = 1
min_wind = 1
max_wind = 11
bins = (1,3,4,5,6,7,8,9,10,11)
bin_labels = ['1-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10','10-11']

rmse_meta = []
rmse_coare = []
rmse_vickers = []
rmse_obs_counts = []

for i in range(len(bins) - 1):
    idx = (X_test[wind_col] >= bins[i]) & (X_test[wind_col] < bins[i+1])
    if np.sum(idx) == 0:
        rmse_meta.append(np.nan)
        rmse_coare.append(np.nan)
        rmse_vickers.append(np.nan)
        rmse_obs_counts.append(0)
        continue
    y_true_bin = y_test[idx]
    rmse_meta.append(np.sqrt(np.mean((meta_pred[idx] - y_true_bin) ** 2)))
    rmse_coare.append(np.sqrt(np.mean((U_star_coare35[idx] - y_true_bin) ** 2)))
    rmse_vickers.append(np.sqrt(np.mean((vickers[idx] - y_true_bin) ** 2)))
    rmse_obs_counts.append(len(y_true_bin))

penguin_means = {
    'ours': rmse_meta,
    'COARE3.5': rmse_coare,
    'Vickers et al(2015)': rmse_vickers
}

colors = {
    'ours': '#1f77b4',       # Blue
    'COARE3.5': '#ff7f0e',   # Orange
    'Vickers et al(2015)': '#2ca02c'  # Green
}

from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(13, 8))
gs = GridSpec(2, 3, figure=fig)

# 第一行: 三个散点图
ax1 = fig.add_subplot(gs[0, 0])  # First row, first column
ax2 = fig.add_subplot(gs[0, 1])  # First row, second column
ax3 = fig.add_subplot(gs[0, 2])  # First row, third column

# 第二行: 合并一个大柱状图
ax4 = fig.add_subplot(gs[1, :])  # Second row, spanning all columns

# 绘制散点图
ax1.scatter(y_test, meta_pred, marker='o', s=18, alpha=0.5, facecolors=colors['ours'],edgecolors='white')
ax1.set_xlabel('Observed $u_*$ (m s$^{-1}$)' ,fontsize=14)
ax1.set_ylabel('Predicted $u_*$ (m s$^{-1}$)',fontsize=14)
ax1.grid(True, which='both', linestyle='--', alpha=0.3)
ax1.set_xlim(0, 0.5)
ax1.set_ylim(0, 0.5)
ax1.xaxis.set_ticks_position('both')  # x轴刻度在底部和顶部
ax1.yaxis.set_ticks_position('both')  # y轴刻度在左侧和右侧
ax1.tick_params(direction='in')
# ax.plot([0, 0.5], [a*0+b, a*0.5+b], 'r--', linewidth=1, alpha=0.5, label='Fitted line')
 # 1:1线
ax1.plot([0, 0.5], [0, 0.5], '--', color='black', linewidth=1, alpha=1, label='1:1 line')

# 线性回归拟合
a, b = np.polyfit(y_test, meta_pred, 1)
ax1.plot([0, 0.5], [a*0+b, a*0.5+b], 'r--', linewidth=1, alpha=0.5, label='Fitted line')
# 评估指标
rmse = np.sqrt(mean_squared_error(y_test, meta_pred))
mape = mean_absolute_percentage_error(y_test, meta_pred) * 100
r2_test = r2_score(y_test, meta_pred)

# 构建文本
text_str = (
    f'y = {a:.3f}x + {b:.3f}\n'
    f'RMSE = {rmse:.3f} m/s\n'
    f'MAPE = {mape:.2f}%\n'
    f'$R^2$ = {r2_test:.2f}\n'
    f'N = {len(y_test)}'
)
# 添加到ax1
ax1.text(0.03, 0.93, text_str,
            transform=ax1.transAxes,
            verticalalignment='top',
            fontsize=13,
            linespacing=1.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5,linewidth=0))


ax2.scatter(y_test, U_star_coare35, marker='o', s=18, alpha=0.5, facecolors=colors['COARE3.5'],edgecolors='white')
ax2.set_xlabel('Observed $u_*$ (m s$^{-1}$)' ,fontsize=14)
ax2.set_ylabel('Predicted $u_*$ (m s$^{-1}$)',fontsize=14)
ax2.grid(True, which='both', linestyle='--', alpha=0.3)
ax2.set_xlim(0, 0.5)
ax2.set_ylim(0, 0.5)
ax2.xaxis.set_ticks_position('both')  # x轴刻度在底部和顶部
ax2.yaxis.set_ticks_position('both')  # y轴刻度在左侧和右侧
ax2.tick_params(direction='in')
ax2.plot([0, 0.5], [0, 0.5], '--', color='black', linewidth=1, alpha=1, label='1:1 line')

# 线性回归拟合
a, b = np.polyfit(y_test, U_star_coare35, 1)
ax2.plot([0, 0.5], [a*0+b, a*0.5+b], 'r--', linewidth=1, alpha=0.5, label='Fitted line')
# 评估指标
rmse = np.sqrt(mean_squared_error(y_test, U_star_coare35))
mape = mean_absolute_percentage_error(y_test, U_star_coare35) * 100
r2_test = r2_score(y_test, U_star_coare35)

# 构建文本
text_str = (
    f'y = {a:.3f}x + {b:.3f}\n'
    f'RMSE = {rmse:.3f} m/s\n'
    f'MAPE = {mape:.2f}%\n'
    f'$R^2$ = {r2_test:.2f}\n'
    f'N = {len(y_test)}'
)
# 添加到ax1
ax2.text(0.03, 0.93, text_str,
            transform=ax2.transAxes,
            verticalalignment='top',
            fontsize=13,
            linespacing=1.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5,linewidth=0))

ax3.scatter(y_test, vickers, marker='o', s=18, alpha=0.5, facecolors=colors['Vickers et al(2015)'],edgecolors='white')
ax3.set_xlabel('Observed $u_*$ (m s$^{-1}$)' ,fontsize=14)
ax3.set_ylabel('Predicted $u_*$ (m s$^{-1}$)',fontsize=14)
ax3.grid(True, which='both', linestyle='--', alpha=0.3)
ax3.set_xlim(0, 0.5)
ax3.set_ylim(0, 0.5)
ax3.xaxis.set_ticks_position('both')  # x轴刻度在底部和顶部
ax3.yaxis.set_ticks_position('both')  # y轴刻度在左侧和右侧
ax3.tick_params(direction='in')
ax3.plot([0, 0.5], [0, 0.5], '--', color='black', linewidth=1, alpha=1, label='1:1 line')

# 线性回归拟合
a, b = np.polyfit(y_test, vickers, 1)
ax3.plot([0, 0.5], [a*0+b, a*0.5+b], 'r--', linewidth=1, alpha=0.5, label='Fitted line')
# 评估指标
rmse = np.sqrt(mean_squared_error(y_test, vickers))
mape = mean_absolute_percentage_error(y_test, vickers) * 100
r2_test = r2_score(y_test, vickers)

# 构建文本
text_str = (
    f'y = {a:.3f}x + {b:.3f}\n'
    f'RMSE = {rmse:.3f} m/s\n'
    f'MAPE = {mape:.2f}%\n'
    f'$R^2$ = {r2_test:.2f}\n'
    f'N = {len(y_test)}'
)
# 添加到ax1
ax3.text(0.03, 0.93, text_str,
            transform=ax3.transAxes,
            verticalalignment='top',
            fontsize=13,
            linespacing=1.5,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5,linewidth=0))
# 绘制合并的柱状图
width = 0.2
multiplier = 0
x = np.arange(len(bin_labels))

for attribute, measurement in penguin_means.items():
    offset = width * multiplier
    rects = ax4.bar(x + offset, measurement, width, label=attribute, zorder=10, color=colors.get(attribute, "#888888"),alpha=0.6)
    multiplier += 1

ax4.set_xticks(x + width)  # 设置x轴刻度位置
ax4.set_xticklabels(bin_labels)  # 设置风速区间标签，并旋转
ax4.set_ylabel('RMSE',fontsize=14)
ax4.set_xlabel('Wind Speed Bin (m s$^{-1}$)' ,fontsize=14)
ax4.grid(which='both', linestyle='--', alpha=0.3, zorder=0)
# 自动调整y轴范围
ax4.set_ylim(0, 0.14)
ax4.legend(ncol=3)

# 计算改进百分比
coare35_array = np.array(penguin_means['COARE3.5'])
ours_array = np.array(penguin_means['ours'])
vickers_array = np.array(penguin_means['Vickers et al(2015)'])

coare35_p = (coare35_array - ours_array) / coare35_array * 100
Vickers_P = (vickers_array - ours_array) / vickers_array * 100

# 在柱状图顶部添加百分比标注
# 确认“ours”这一组的柱子索引，默认和“COARE3.5”,“Vickers et al(2015)”一致
ours_idx = list(penguin_means.keys()).index('ours')
coare35_idx = list(penguin_means.keys()).index('COARE3.5')
vickers_idx = list(penguin_means.keys()).index('Vickers et al(2015)')

for i, (c35, vic) in enumerate(zip(coare35_p, Vickers_P)):
    # COARE3.5柱的位置
    x_coare = x[i] + width * coare35_idx-0.1
    y_coare = coare35_array[i]
    # Vickers柱的位置
    x_vickers = x[i] + width * vickers_idx
    y_vickers = vickers_array[i]
    
    # 分别对COARE3.5和Vickers的顶部写出 ours 相对于他们的下降百分比，向上偏移更美观
    # 使用自定义unicode符号"↓"和"⬇"效果更佳，或尝试matplotlib数学表达式（如 r'$\downarrow$'）
    ax4.text(x_coare-0.02, y_coare + 0.002, f'↓{c35:.1f}%', ha='center', va='bottom', fontsize=11, color=colors.get('COARE3.5', "#888888"))
    ax4.text(x_vickers+0.06, y_vickers + 0.002, f'↓{vic:.1f}%', ha='center', va='bottom', fontsize=11, color=colors.get('Vickers et al(2015)', "#888888"))


ax1.text(0.01, 0.95, r'$\mathbf{(a)}$', transform=ax1.transAxes,fontsize=14)
ax2.text(0.01, 0.95, r'$\mathbf{(b)}$', transform=ax2.transAxes,fontsize=14)
ax3.text(0.01, 0.95, r'$\mathbf{(c)}$', transform=ax3.transAxes,fontsize=14)
ax4.text(0.005, 0.95, r'$\mathbf{(d)}$', transform=ax4.transAxes,fontsize=14)

ax1.text(0.5, 0.1, r'$\mathbf{ours}$', transform=ax1.transAxes, fontsize=14)
ax2.text(0.5, 0.1, r'$\mathbf{COARE3.5}$', transform=ax2.transAxes, fontsize=14)
ax3.text(0.5, 0.1, r'$\mathbf{Vickers\ et\ al(2015)}$', transform=ax3.transAxes, fontsize=14)

plt.tight_layout()
plt.savefig("/share/home/jinhm/PhD_Project/B7_WAVE_U/image/meta_metrics.tiff", dpi=300)
plt.show()


# 统一保存所有长度一样的数据到一个CSV，因为它们的长度完全一致
# bin_labels, penguin_means每个方法, rmse_obs_counts 合并成一个DataFrame

result_df = pd.DataFrame({'bin': bin_labels})

# penguin_means每个方法
for key in penguin_means:
    result_df[key] = np.round(penguin_means[key], 3)

result_df['VS coare35'] = np.round(coare35_p, 1)
result_df['VS Vickers'] = np.round(Vickers_P, 1)
# rmse_obs_counts
result_df['count'] = rmse_obs_counts

# 保存到一个CSV文件中
result_df.to_csv('/share/home/jinhm/PhD_Project/B7_WAVE_U/image/metrics_bin_summary.csv', index=False)

all_coare = np.sqrt((mean_squared_error(y_test, U_star_coare35)))
all_vickers = np.sqrt((mean_squared_error(y_test, vickers)))
all_ours = np.sqrt((mean_squared_error(y_test, meta_pred)))

print((all_coare-all_ours)/all_coare*100)
print((all_vickers-all_ours)/all_vickers*100)

print(all_coare,all_vickers,all_ours)



In [ ]:
# 评估第二层模型的精度
import matplotlib.pyplot as plt
import matplotlib as mpl

models_name_dict = {
    'xgboost': 'XGBoost',
    'lightgbm': 'LightGBM',
    'random_forest': 'Random Forest',
    'adaboost': 'AdaBoost',
    'catboost': 'CatBoost',
    'svm': 'SVR',
    'gbdt': 'GBDT',
    'cart': 'CART',
    'plsr': 'PLSR',
    'mlp': 'MLP',
}

# 增加集合模型（第二层meta）
ensemble_key = 'ours'
models_name_dict_with_ensemble = models_name_dict.copy()
models_name_dict_with_ensemble[ensemble_key] = 'HuberRegressor'

data = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/DataSet.xlsx')
X = data[['u10','Hs', 'direction','wave_steepnessdou', 'B*', 'cp', 'kp', 'fp_comp']]
y = data['u_star_down']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

wind_col = 'u10'
bin_width = 1
bins = (1,3,4,5,6,7,8,9,10,11)
bin_labels = ['1-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10','10-11']
print(bin_labels)

samples_per_bin = []
model_rmse_per_bin = {name: [] for name in models_name_dict_with_ensemble.keys()}

first_level_preds = {}
for name in models_name_dict.keys():
    model_path = f'/share/home/jinhm/PhD_Project/B7_WAVE_U/result/{name}/best_model.pkl'
    model = joblib.load(model_path)
    first_level_preds[name] = model.predict(X_test)

meta_model = joblib.load('/share/home/jinhm/PhD_Project/B7_WAVE_U/result/HuberRegressor/best_model.pkl')

for i in range(len(bins) - 1):
    idx = (X_test[wind_col] >= bins[i]) & (X_test[wind_col] < bins[i + 1])
    sample_num = idx.sum()
    samples_per_bin.append(sample_num)
    if sample_num < 5:
        for name in models_name_dict_with_ensemble.keys():
            model_rmse_per_bin[name].append(np.nan)
        continue
    y_true_bin = y_test[idx]
    for name in models_name_dict.keys():
        y_pred_bin = first_level_preds[name][idx]
        try:
            score = np.sqrt(mean_squared_error(y_true_bin, y_pred_bin))
        except Exception:
            score = np.nan
        model_rmse_per_bin[name].append(score)
    meta_input_bin = np.column_stack([first_level_preds[name][idx] for name in models_name_dict.keys()])
    try:
        y_meta_pred_bin = meta_model.predict(meta_input_bin)
        meta_score = np.sqrt(mean_squared_error(y_true_bin, y_meta_pred_bin))
    except Exception:
        meta_score = np.nan
    model_rmse_per_bin[ensemble_key].append(meta_score)

# ----------- 绘图润色部分 -----------
hero_color = "#111111"
palette = plt.get_cmap("tab10")
fig, ax = plt.subplots(figsize=(10, 5))  # ≈Nature 1-column width

all_keys = list(models_name_dict.keys()) + [ensemble_key]

# 画一级模型
for idx, name in enumerate(models_name_dict_with_ensemble.keys()):
    if name == 'ours':
        ax.plot(
            bin_labels, model_rmse_per_bin[ensemble_key],
            marker='*', linestyle='-', linewidth=2.1, markersize=9,
            label='ours',
            color=hero_color, alpha=0.95, zorder=10,
            markeredgewidth=1, markeredgecolor=hero_color
        )
    else:
        ax.plot(
        bin_labels, model_rmse_per_bin[name],
        marker='o', linestyle='--', markersize=4.2,
        label=models_name_dict[name],
        color=palette(idx), linewidth=1,
        alpha=1, zorder=2)

ax.set_ylim(0.01, 0.08)
ax.set_yticks(np.arange(0.01, 0.09, 0.01))

ax.set_xlabel('Wind Speed Bin (m s$^{-1}$)')
ax.set_ylabel('RMSE')
ax.tick_params(axis='x')

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("black")

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=6, frameon=False,
    fontsize=10,
    handletextpad=0.5,
    columnspacing=1.2
)
ax.grid(True, linestyle='--', linewidth=0.55, alpha=0.55, zorder=1)
ax.set_axisbelow(True)

# 区间样本数标注 (上移些距离，避免覆盖)
ymax = np.nanmax([np.nanmax(vals) for vals in model_rmse_per_bin.values() if isinstance(vals, list) and vals])
for i, label in enumerate(bin_labels):
    ax.annotate(
        f'n={samples_per_bin[i]}',
        xy=(i, ymax * 1.01),
        xytext=(0, 0),
        textcoords='offset points',
        ha='center', va='bottom',
        fontsize=10,
        color='dimgray',
        clip_on=False
    )

fig.tight_layout(rect=[0, 0.14, 1, 1])

# 推荐导出矢量+高清格式（见nature-figure SKILL）
def save_pub_py(fig, filename, dpi=300):
    # fig.savefig(f"{filename}.svg", bbox_inches="tight")
    # fig.savefig(f"{filename}.pdf", bbox_inches="tight")
    fig.savefig(f"{filename}.tiff", dpi=dpi, bbox_inches="tight")

# 如需保存图像启用以下行
save_pub_py(fig, "/share/home/jinhm/PhD_Project/B7_WAVE_U/image/rmse_per_bin")
plt.show()

In [ ]:
# 分析系数
# meta_model = joblib.load('/share/home/jinhm/PhD_Project/B7_WAVE_U/result/HuberRegressor/best_model.pkl')
# models_name = (
#     'XGBoost',
#     'LightGBM',
#     'Random Forest',
#     'AdaBoost',
#     'CatBoost',
#     'SVM',
#     'GBDT',
#     'CART',
#     'PLSR',
#     'MLP'
# )

# coefficients = pd.DataFrame({
#     'model': models_name,
#     'coefficient': meta_model.coef_,
#     'importance_pct': np.abs(meta_model.coef_) / np.sum(np.abs(meta_model.coef_)) * 100
# })

# coefficients.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/base_importance.xlsx',index=False)
coefficients = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/base_importance.xlsx')

# ========== 图2：散点图（带重要性大小映射）==========
fig, ax = plt.subplots(figsize=(10, 4))

# 为每个模型设置位置
x_positions = np.arange(len(coefficients))
y = coefficients['importance_pct'].to_numpy()

# 决策树
ax.plot(x_positions[0], y[0], 'o', markersize=5, color='#DC267F', label='XGBoost')
ax.plot(x_positions[1], y[1], 'o', markersize=5, color='#DC267F', label='LightGBM')
ax.plot(x_positions[2], y[2], 'o', markersize=5, color='#DC267F', label='Random Forest')
ax.plot(x_positions[3], y[3], 'o', markersize=5, color='#DC267F', label='AdaBoost')
ax.plot(x_positions[4], y[4], 'o', markersize=5, color='#DC267F', label='CatBoost')
ax.plot(x_positions[7], y[7], 'o', markersize=5, color='#DC267F', label='CART')
ax.plot(x_positions[6], y[6], 'o', markersize=5, color='#DC267F', label='GBDT')
# markerfacecolor='none', markeredgewidth=1
ax.plot(x_positions[0], y[0], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[1], y[1], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[2], y[2], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[3], y[3], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[4], y[4], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[7], y[7], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.plot(x_positions[6], y[6], 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)

ax.text(x_positions[0]+0.1,y[0]+1.0,f'{y[0]:.2f}%',ha='center')
ax.text(x_positions[1]+0.1,y[1]+1.0,f'{y[1]:.2f}%',ha='center')
ax.text(x_positions[2]+0.1,y[2]+1.0,f'{y[2]:.2f}%',ha='center')
ax.text(x_positions[3]+0.1,y[3]+1.0,f'{y[3]:.2f}%',ha='center')
ax.text(x_positions[4]+0.1,y[4]+1.0,f'{y[4]:.2f}%',ha='center')
ax.text(x_positions[7]+0.1,y[7]+1.0,f'{y[7]:.2f}%',ha='center')
ax.text(x_positions[6]+0.1,y[6]+1.0,f'{y[6]:.2f}%',ha='center')
# 核方法
ax.plot(x_positions[5], y[5]-0.15, '^', markersize=4, color='#009E73', label='SVR')
ax.plot(x_positions[5], y[5], '^', markersize=9, color='#009E73', label='SVR', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[5]+0.1,y[5]+1.0,f'{y[5]:.2f}%',ha='center')
# 线形模型
# 改成正方形
ax.plot(x_positions[8], y[8], 's', markersize=6, color='#0072B2', label='PLSR')  # 's' for square marker
ax.plot(x_positions[8], y[8], 's', markersize=9, color='#0072B2', label='PLSR', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[8]+0.1,y[8]+1.0,f'{y[8]:.2f}%',ha='center')
# 神经网络
ax.plot(x_positions[9], y[9], 'D', markersize=5, color='#D55E00', label='MLP')
ax.plot(x_positions[9], y[9], 'D', markersize=9, color='#D55E00', label='MLP',markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[9]+0.1,y[9]+1.0,f'{y[9]:.2f}%',ha='center')


# 添加图例
a = 29.8
ax.plot(x_positions[1], a, 'o', markersize=5, color='#DC267F')
ax.plot(x_positions[1], a, 'o', markersize=9, color='#DC267F', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[1]+0.15, a-0.5, f'Decision Trees',ha='left')

ax.plot(x_positions[3], a-0.15, '^', markersize=4, color='#009E73')
ax.plot(x_positions[3], a, '^', markersize=9, color='#009E73', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[3]+0.15, a-0.5, f'Kernel Methods',ha='left')

ax.plot(x_positions[5], a, 's', markersize=5, color='#0072B2')
ax.plot(x_positions[5], a, 's', markersize=9, color='#0072B2', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[5]+0.15, a-0.5, f'Linear Models',ha='left')

ax.plot(x_positions[7], a, 'D', markersize=5, color='#D55E00')  # 'D' for diamond marker
ax.plot(x_positions[7], a, 'D', markersize=9, color='#D55E00', markerfacecolor='none', markeredgewidth=1)
ax.text(x_positions[7]+0.15, a-0.5, f'Neural Networks',ha='left')

# 添加标签
ax.set_xticks(x_positions)
ax.set_xticklabels(coefficients['model'].to_numpy(), rotation=0, ha='center',fontsize=10)
ax.set_xlim(-0.5,9.5)
ax.set_yticks(np.arange(0, 34, 4))
ax.set_xlabel('Base learners')
ax.set_ylabel('Contribution Rate(%)', fontsize=12)

# 添加网格
ax.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('/share/home/jinhm/PhD_Project/B7_WAVE_U/image/base_importance_scatter.tiff', dpi=300, bbox_inches='tight')
plt.show()